In [5]:
from google.colab import files
files.upload()

Saving q3_retail_promotions.csv to q3_retail_promotions (1).csv


{'q3_retail_promotions (1).csv': b'transaction_date,store_id,store_size,location_type,promotion_type,is_weekend,is_festival,competition_density,items_sold\n2022-01-01,28,small,semi-urban,free_gift,1,0,5,224\n2022-01-01,5,medium,semi-urban,free_gift,1,1,1,348\n2022-01-02,13,small,semi-urban,loyalty_points,1,0,6,249\n2022-01-02,17,small,urban,free_gift,1,0,7,259\n2022-01-03,50,medium,semi-urban,bogo,0,0,3,277\n2022-01-03,40,large,urban,category_offer,0,1,4,411\n2022-01-04,45,small,semi-urban,free_gift,0,0,3,193\n2022-01-04,32,medium,rural,flat_discount,0,0,1,240\n2022-01-05,29,large,urban,free_gift,0,0,8,305\n2022-01-05,3,small,rural,loyalty_points,0,0,2,181\n2022-01-06,40,medium,urban,bogo,0,0,1,331\n2022-01-07,34,medium,semi-urban,bogo,0,0,8,255\n2022-01-08,40,medium,rural,flat_discount,1,0,7,270\n2022-01-09,27,small,urban,flat_discount,1,1,7,359\n2022-01-10,13,medium,urban,bogo,0,0,4,271\n2022-01-10,45,large,semi-urban,free_gift,0,0,9,274\n2022-01-11,33,medium,urban,flat_discount,0,0,

In [6]:
import pandas as pd

df = pd.read_csv("q3_retail_promotions.csv")
df.head()

,transaction_date,store_id,store_size,location_type,promotion_type,is_weekend,is_festival,competition_density,items_sold
0,2022-01-01,28,small,semi-urban,free_gift,1,0,5,224
1,2022-01-01,5,medium,semi-urban,free_gift,1,1,1,348
2,2022-01-02,13,small,semi-urban,loyalty_points,1,0,6,249
3,2022-01-02,17,small,urban,free_gift,1,0,7,259
4,2022-01-03,50,medium,semi-urban,bogo,0,0,3,277


In [7]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

In [8]:
df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.dayofweek

In [9]:
df['is_month_end'] = df['transaction_date'].dt.day >= 25
df['is_month_end'] = df['is_month_end'].astype(int)

Date features such as year, month, and day_of_week are extracted.
A new feature is_month_end is created to capture end-of-month effects.

In [10]:
df = df.sort_values(by='transaction_date')

In [11]:
train_size = int(0.8 * len(df))

train = df.iloc[:train_size]
test = df.iloc[train_size:]

Data is split based on time to preserve chronological order, as random splitting is not suitable for time-series data.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Features & target
X_train = train.drop('items_sold', axis=1)
y_train = train['items_sold']

X_test = test.drop('items_sold', axis=1)
y_test = test['items_sold']

# Drop date column
X_train = X_train.drop('transaction_date', axis=1)
X_test = X_test.drop('transaction_date', axis=1)

# Column types
categorical_cols = ['promotion_type', 'location_type', 'store_size']
numerical_cols = [col for col in X_train.columns if col not in categorical_cols]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first'), categorical_cols)
    ]
)

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Linear Regression Pipeline
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# Random Forest Pipeline
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

# Train
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['store_id', 'is_weekend',
                                                   'is_festival',
                                                   'competition_density',
                                                   'year', 'month',
                                                   'day_of_week',
                                                   'is_month_end']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['promotion_type',
                                                   'location_type',
                                                   'store_size'])])),
                ('model', RandomForestRegressor(random_state=42))])

In [15]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

print("Linear Regression:")
print("RMSE:", (mean_squared_error(y_test, lr_pred)) ** 0.5)
print("MAE:", mean_absolute_error(y_test, lr_pred))

print("\nRandom Forest:")
print("RMSE:", (mean_squared_error(y_test, rf_pred)) ** 0.5)
print("MAE:", mean_absolute_error(y_test, rf_pred))

Linear Regression:
RMSE: 27.12145116489062
MAE: 21.052926674588395

Random Forest:
RMSE: 31.658897860633115
MAE: 24.904708333333335


Models are evaluated using RMSE and MAE. Lower values indicate better performance.